In [ ]:
import io
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

In [ ]:
# AMAÇ:
# Bir ses kaydındaki veriler iki kümeye ayrılıyor mu?
# Ve bu iki küme gerçekten PD ve Healthy mi?

In [ ]:
# 4 ölçüm yaptık
# Centroid Distance (Merkezler arası mesafe): 2 kümeye ayırıp bu küme merkezleri arasındaki mesafe ölçüldü. Büyük ise iki küme birbirinden uzak, küçükse kümeler yakın.
# Within-Cluster Spread (Küme içi yayılım): Her noktanın kendi küme merkezine olan ortalama uzaklığı. Küçükse küme sıkı, büyükse küme dağınık.
# Separation ratio (Centroid Distance/within spread):  Merkezler arası mesafe / küme içi dağınıklık
# Separation ratio, "İki küme birbirinden ne kadar net ayrılmış?" ölçer.
# Separation ratio, büyük ise iki küme net şekilde ayrılmıştır, küçük ise iki küme birbirine karışmış gibi düşün

# ARI: KMeans veriyi 2 kümeye ayırdı: Küme 0, Küme 1

# Gerçek etiketler de: Healthy (0), PD (1)

# ARI şunu sorar: Küme 0 gerçekten Healthy mi?, Küme 1 gerçekten PD mi?
# -1 ile +1 arasında değerler alır.
# 1 : güçlü ayrım, -1: zayıf ayrım

# Eğer büyük ölçüde örtüşüyorsa ARI yükselir.

# Ratio şunu ölçer: Veri iki kümeye ayrılıyor mu?
# ARI şunu ölçer: Bu iki küme gerçekten PD–Healthy mi?

# Standard Scaler: Verideki feature'ların yeni ortalaması 0, standart sapması 1 olacak şekilde ölçekleme yapar.
# Sebebi büyük değerlerin küçük değerleri bastırmasını engellemektir.

In [ ]:
from google.colab import files

uploaded = files.upload()  # train_data.txt seç
train_name = list(uploaded.keys())[0]

df_train = pd.read_csv(io.BytesIO(uploaded[train_name]), header=None)

print("Yüklenen train:", train_name)
print("Train shape:", df_train.shape)  # (1040, 29)

Saving train_data.txt to train_data (2).txt
Yüklenen train: train_data (2).txt
Train shape: (1040, 29)


In [ ]:
# Sütunlar:
# 0: subject_id
# 1-26: features (26 adet)
# 27: UPDRS
# 28: class

print("Unique class:", sorted(df_train.iloc[:, 28].unique()))
print("Subject sayısı:", df_train.iloc[:, 0].nunique())

# 26'lık blok kontrolü: ilk 30 satırın subject id'si aynı mı?
print("İlk 30 subject id:", df_train.iloc[:30, 0].tolist())

Unique class: [np.int64(0), np.int64(1)]
Subject sayısı: 40
İlk 30 subject id: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2]


In [ ]:
num_records = 26

recording_types = {
    1: "aaa",
    2: "ooo",
    3: "uuu",
    4: "number_1",
    5: "number_2",
    6: "number_3",
    7: "number_4",
    8: "number_5",
    9: "number_6",
    10: "number_7",
    11: "number_8",
    12: "number_9",
    13: "number_10",
    14: "sentence_1",
    15: "sentence_2",
    16: "sentence_3",
    17: "sentence_4",
    18: "word_1",
    19: "word_2",
    20: "word_3",
    21: "word_4",
    22: "word_5",
    23: "word_6",
    24: "word_7",
    25: "word_8",
    26: "word_9",
}

os.makedirs("recordings_csv", exist_ok=True)
os.makedirs("tsne_png", exist_ok=True)

print("Klasörler hazır: recordings_csv/, tsne_png/")

Klasörler hazır: recordings_csv/, tsne_png/


In [ ]:
# 26 ayrı dosyaya bolmek

In [ ]:
for i in range(num_records):
    rec = df_train.iloc[i::26, :].copy()
    fname = f"recordings_csv/recording_{i+1:02d}_{recording_types[i+1]}.csv"
    rec.to_csv(fname, index=False)

print("26 recording CSV kaydedildi.")

26 recording CSV kaydedildi.


In [ ]:
results = []

for i in range(num_records):
    rec_no = i + 1
    rec_type = recording_types[rec_no]
    csv_path = f"recordings_csv/recording_{rec_no:02d}_{rec_type}.csv"

    print(f"[{rec_no:02d}/26] Başlıyor: {rec_type}")

    try:
        rec = pd.read_csv(csv_path)

        # X: 26 feature (col 2-27 => index 1:27)
        X = rec.iloc[:, 1:27].values

        # y: class (col 29 => index 28)
        y = rec.iloc[:, 28].values

        # 1) StandardScaler
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        # 2) t-SNE (sadece görsel)
        # 40 örnek için perplexity 5-15 aralığı uygundur
        tsne = TSNE(n_components=2, perplexity=10, random_state=42, init="pca")
        X_2d = tsne.fit_transform(X_scaled)

        plt.figure(figsize=(5, 4))

        classes = np.unique(y)
        for cls in classes:
            idx = (y == cls)
            plt.scatter(X_2d[idx, 0], X_2d[idx, 1], label=f"Class {cls}", s=25)

        plt.title(f"t-SNE | {rec_no:02d} - {rec_type}")
        plt.legend(loc="best", fontsize=8)
        plt.tight_layout()

        png_path = f"tsne_png/tsne_recording_{rec_no:02d}_{rec_type}.png"
        plt.savefig(png_path, dpi=150)
        plt.close()


        # 3) KMeans (analiz)
        kmeans = KMeans(n_clusters=2, n_init=30, random_state=42)
        clusters = kmeans.fit_predict(X_scaled)

        centroids = kmeans.cluster_centers_

        # centroid distance
        centroid_distance = np.linalg.norm(centroids[0] - centroids[1])

        # within-cluster spread (tüm noktaların kendi merkezine uzaklığının ortalaması)
        dists = np.linalg.norm(X_scaled - centroids[clusters], axis=1)
        spread = np.mean(dists)
        # ratio
        ratio = centroid_distance / spread if spread != 0 else np.nan

        # ARI
        ari = adjusted_rand_score(y, clusters)

        results.append([rec_no, rec_type, centroid_distance, spread, ratio, ari])

        print(f"Bitti | ARI={ari:.4f} | Ratio={ratio:.4f}")

    except Exception as e:
        print(f"HATA ({rec_no:02d} - {rec_type}): {e}")
        # Devam etsin diye break koymuyoruz; hata olsa da diğer recording'lere geçer
        continue

print("Analiz döngüsü tamamlandı.")

[01/26] Başlıyor: aaa
Bitti | ARI=-0.0065 | Ratio=1.9603
[02/26] Başlıyor: ooo
Bitti | ARI=-0.0088 | Ratio=2.4170
[03/26] Başlıyor: uuu
Bitti | ARI=0.0047 | Ratio=1.8194
[04/26] Başlıyor: number_1
Bitti | ARI=0.0021 | Ratio=1.3999
[05/26] Başlıyor: number_2
Bitti | ARI=-0.0133 | Ratio=1.7163
[06/26] Başlıyor: number_3
Bitti | ARI=0.0409 | Ratio=1.4546
[07/26] Başlıyor: number_4
Bitti | ARI=-0.0088 | Ratio=1.8882
[08/26] Başlıyor: number_5
Bitti | ARI=-0.0118 | Ratio=2.0542
[09/26] Başlıyor: number_6
Bitti | ARI=-0.0031 | Ratio=2.0010
[10/26] Başlıyor: number_7
Bitti | ARI=0.0000 | Ratio=4.1338
[11/26] Başlıyor: number_8
Bitti | ARI=-0.0125 | Ratio=1.9987
[12/26] Başlıyor: number_9
Bitti | ARI=0.0052 | Ratio=3.0456
[13/26] Başlıyor: number_10
Bitti | ARI=-0.0221 | Ratio=1.1523
[14/26] Başlıyor: sentence_1
Bitti | ARI=-0.0157 | Ratio=1.6488
[15/26] Başlıyor: sentence_2
Bitti | ARI=-0.0133 | Ratio=2.1793
[16/26] Başlıyor: sentence_3
Bitti | ARI=-0.0221 | Ratio=1.3324
[17/26] Başlıyor: sen

In [ ]:
results_df = pd.DataFrame(
    results,
    columns=["Recording_No", "Type", "Centroid_Distance", "Within_Spread", "Separation_Ratio", "ARI"]
)

results_df_sorted = results_df.sort_values(by=["ARI", "Separation_Ratio"], ascending=False)

print("Toplam analiz edilen recording sayısı:", results_df.shape[0])
print(results_df_sorted)

results_df_sorted.to_csv("record_analysis_results.csv", index=False)
print("Kaydedildi: record_analysis_results.csv")

Toplam analiz edilen recording sayısı: 26
    Recording_No        Type  Centroid_Distance  Within_Spread  \
5              6    number_3           5.797372       3.985605   
11            12    number_9          12.278171       4.031426   
2              3         uuu           7.083751       3.893420   
3              4    number_1           5.827071       4.162575   
9             10    number_7          16.549373       4.003416   
23            24      word_7           5.048992       4.280802   
17            18      word_1           4.888782       4.146208   
24            25      word_8           4.875233       4.166158   
8              9    number_6           7.873124       3.934675   
19            20      word_3           5.610520       4.047862   
0              1         aaa           7.381353       3.765496   
1              2         ooo           9.209169       3.810138   
6              7    number_4           7.797781       4.129784   
7              8    number_5      

In [ ]:
png_files = sorted([f for f in os.listdir("tsne_png") if f.endswith(".png")])
csv_files = sorted([f for f in os.listdir("recordings_csv") if f.endswith(".csv")])

print("PNG sayısı:", len(png_files))
print("CSV sayısı:", len(csv_files))

print("Eksik PNG var mı?")
expected = {f"tsne_recording_{i:02d}_{recording_types[i]}.png" for i in range(1, 27)}
actual = set(png_files)
missing = sorted(list(expected - actual))
print(missing[:10], "..." if len(missing) > 10 else "")

PNG sayısı: 26
CSV sayısı: 26
Eksik PNG var mı?
[] 


In [ ]:
import shutil, os
from google.colab import files

OUT_DIR = "analysis_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# Çıktı dosyalarını tek klasörde topla
shutil.copy("record_analysis_results.csv", os.path.join(OUT_DIR, "record_analysis_results.csv"))

# Klasörleri kopyala (varsa üzerine yazar)
if os.path.exists("tsne_png"):
    shutil.copytree("tsne_png", os.path.join(OUT_DIR, "tsne_png"), dirs_exist_ok=True)

if os.path.exists("recordings_csv"):
    shutil.copytree("recordings_csv", os.path.join(OUT_DIR, "recordings_csv"), dirs_exist_ok=True)

# Sadece bu klasörü zip'le
shutil.make_archive("analysis_output_update", "zip", OUT_DIR)
files.download("analysis_output_update.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>